# mycx_1000 ML validation workbench

This notebook is a lightweight smoke-test path for offline replay training.
Use it to validate cache, snapshot plans, and optimizer wiring before long formal runs.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "config.py").exists():
    candidates = [REPO_ROOT.parent, REPO_ROOT.parent.parent]
    for candidate in candidates:
        if (candidate / "config.py").exists():
            REPO_ROOT = candidate
            break

if not (REPO_ROOT / "config.py").exists():
    raise RuntimeError("Could not locate repo root. Open the notebook from the mycx_1000 workspace.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT


In [ ]:
import importlib
import pandas as pd
import matplotlib.pyplot as plt

from config import DEFAULT_CONFIG
import tuner.data_cache as data_cache_mod
import tuner.offline_evaluator as offline_evaluator_mod
import tuner.train as train_mod

# Force reload so the notebook is safe even if this kernel imported old tuner modules earlier.
data_cache_mod = importlib.reload(data_cache_mod)
offline_evaluator_mod = importlib.reload(offline_evaluator_mod)
train_mod = importlib.reload(train_mod)

CACHE_ROOT = data_cache_mod.CACHE_ROOT
cache_historical_events = data_cache_mod.cache_historical_events
OfflineEvaluator = offline_evaluator_mod.OfflineEvaluator
build_snapshot_plan = train_mod.build_snapshot_plan
run_training = train_mod.run_training
split_event_ids = train_mod.split_event_ids
split_formal_event_ids = train_mod.split_formal_event_ids
split_formal_training_event_ids = train_mod.split_formal_training_event_ids

plt.style.use("ggplot")


## 1. Configure the workbench

If cache is empty or not large enough, the notebook will auto-fetch history snapshots.


In [ ]:
CACHE_DIR = CACHE_ROOT
HISTORY_COUNT = 100
MIN_ELIGIBLE_EVENTS = 12
MIN_TEST_EVENTS = 4
SNAPSHOT_PROGRESS_BUCKETS = [
    (0.15, 0.25),
    (0.25, 0.40),
    (0.40, 0.55),
    (0.55, 0.70),
    (0.70, 0.85),
]
SNAPSHOT_SAMPLES_PER_BUCKET = 1
SNAPSHOT_SEED = 42
PREPARE_CACHE = False
REFRESH_CACHE = False
TRAIN_LIMIT = 8
TEST_LIMIT = 4

# Training runtime controls
MAXITER = 2
POPSIZE = 4
VERBOSE_TRAIN = True
LOG_EVERY_EVALS = 20
TRAIN_WORKERS = 1  # Scale this up after the notebook path is validated on your machine.
USE_GPU = False  # Current run_training path is CPU-only.

BASE_CONFIG = DEFAULT_CONFIG.copy()
BASE_CONFIG["similar_count"] = 5
BASE_CONFIG["ignore_event_ids"] = list(DEFAULT_CONFIG["ignore_event_ids"])

# Notebook validation defaults:
# - Use deterministic random snapshots instead of a few fixed debug hours
# - Keep enough holdout events to assess generalization
# - Avoid aggressive manual train/test truncation

In [ ]:
def compute_eligible_event_ids(evaluator, similar_count, ignore_ids):
    if hasattr(evaluator, "eligible_event_ids"):
        return evaluator.eligible_event_ids(similar_count=similar_count, ignore_ids=ignore_ids)

    # Compatibility fallback for stale kernels or partially reloaded modules.
    eligible = []
    for event_id in evaluator.available_event_ids():
        cached_event = evaluator.cached_events[event_id]
        event_type = cached_event.get("event_type") or cached_event.get("meta", {}).get("event_type", "unknown")
        history_ids = evaluator._select_history_event_ids(
            target_event_id=event_id,
            event_type=event_type,
            count=similar_count,
            ignore_ids=ignore_ids,
        )
        if len(history_ids) >= similar_count:
            eligible.append(event_id)
    return eligible

def load_cache_state():
    evaluator = OfflineEvaluator.from_cache_dir(cache_root=CACHE_DIR)
    event_ids = evaluator.available_event_ids()
    eligible_ids = compute_eligible_event_ids(
        evaluator,
        similar_count=BASE_CONFIG["similar_count"],
        ignore_ids=BASE_CONFIG.get("ignore_event_ids", []),
    )
    return evaluator, event_ids, eligible_ids

evaluator, event_ids, eligible_ids = load_cache_state()
need_prepare = PREPARE_CACHE or len(eligible_ids) < MIN_ELIGIBLE_EVENTS

if need_prepare:
    cache_summary = cache_historical_events(
        history_count=HISTORY_COUNT,
        cache_root=CACHE_DIR,
        refresh=REFRESH_CACHE,
        api_source=BASE_CONFIG["api_source"],
    )
    evaluator, event_ids, eligible_ids = load_cache_state()
else:
    cache_summary = {
        "cache_root": str(CACHE_DIR),
        "prepared": False,
        "cached_events": len(event_ids),
        "eligible_events": len(eligible_ids),
    }

cache_summary


In [ ]:
train_ids, test_ids, external_holdout_ids = split_formal_training_event_ids(
    eligible_ids,
    train_ratio=0.75,
    evaluator=evaluator,
    min_test_events=MIN_TEST_EVENTS,
    min_event_id=200,
    train_max_event_id=300,
    holdout_min_event_id=301,
)

if TRAIN_LIMIT is not None:
    train_ids = train_ids[-int(TRAIN_LIMIT):]
if TEST_LIMIT is not None:
    test_ids = test_ids[:int(TEST_LIMIT)]

train_snapshot_plan = build_snapshot_plan(
    evaluator,
    train_ids,
    seed=SNAPSHOT_SEED,
    progress_buckets=SNAPSHOT_PROGRESS_BUCKETS,
    samples_per_bucket=SNAPSHOT_SAMPLES_PER_BUCKET,
)
test_snapshot_plan = build_snapshot_plan(
    evaluator,
    test_ids,
    seed=SNAPSHOT_SEED + 9973,
    progress_buckets=SNAPSHOT_PROGRESS_BUCKETS,
    samples_per_bucket=SNAPSHOT_SAMPLES_PER_BUCKET,
)
external_holdout_snapshot_plan = build_snapshot_plan(
    evaluator,
    external_holdout_ids,
    seed=SNAPSHOT_SEED + 19973,
    progress_buckets=SNAPSHOT_PROGRESS_BUCKETS,
    samples_per_bucket=SNAPSHOT_SAMPLES_PER_BUCKET,
)

print(f"cached events: {len(event_ids)}")
print(f"eligible events: {len(eligible_ids)}")
print(f"snapshot buckets: {SNAPSHOT_PROGRESS_BUCKETS}")
print(f"train snapshots: {sum(len(v) for v in train_snapshot_plan.values())} | validation snapshots: {sum(len(v) for v in test_snapshot_plan.values())}")
print(f"external holdout snapshots: {sum(len(v) for v in external_holdout_snapshot_plan.values())}")
print(f"formal split: train/validation<=300, external holdout>=301")
print(f"train ids: {len(train_ids)} | validation ids: {len(test_ids)} | external holdout ids: {len(external_holdout_ids)}")

if len(train_ids) < 1 or len(test_ids) < 1:
    raise RuntimeError(
        "Not enough train/test events for validation. Increase HISTORY_COUNT or lower BASE_CONFIG['similar_count']."
    )

{
    "eligible_preview": eligible_ids[:10],
    "train_snapshot_example": {event_id: train_snapshot_plan[event_id] for event_id in train_ids[:2]},
    "test_snapshot_example": {event_id: test_snapshot_plan[event_id] for event_id in test_ids[:2]},
    "external_holdout_snapshot_example": {event_id: external_holdout_snapshot_plan[event_id] for event_id in external_holdout_ids[:2]},
}

## 2. Baseline snapshot

Inspect the default config before optimization.


In [ ]:
baseline_metrics = evaluator.evaluate(
    BASE_CONFIG,
    event_ids=test_ids if test_ids else train_ids,
    snapshot_plan=test_snapshot_plan if test_ids else train_snapshot_plan,
)

baseline_df = pd.DataFrame(baseline_metrics["rows"])
baseline_df.head()


In [ ]:
baseline_metrics


## 3. Run a small training pass

Start small, confirm the loop works, then scale `maxiter` and `popsize` up.


In [ ]:
snapshot_plan = {}
snapshot_plan.update(train_snapshot_plan)
snapshot_plan.update(test_snapshot_plan)

result = run_training(
    cache_root=CACHE_DIR,
    prepare_cache=False,
    train_ids=train_ids,
    test_ids=test_ids,
    snapshot_plan=snapshot_plan,
    snapshot_progress_buckets=SNAPSHOT_PROGRESS_BUCKETS,
    snapshot_samples_per_bucket=SNAPSHOT_SAMPLES_PER_BUCKET,
    history_count=HISTORY_COUNT,
    min_test_events=MIN_TEST_EVENTS,
    use_formal_holdout_split=True,
    formal_min_event_id=200,
    formal_train_max_event_id=300,
    formal_holdout_min_event_id=301,
    external_holdout_ids=external_holdout_ids,
    external_holdout_snapshot_plan=external_holdout_snapshot_plan,
    maxiter=MAXITER,
    popsize=POPSIZE,
    seed=SNAPSHOT_SEED,
    base_config=BASE_CONFIG,
    output_preset_path=REPO_ROOT / "configs" / "models" / "skeleton_kf" / "learned_notebook.json",
    verbose=VERBOSE_TRAIN,
    log_every_n_evals=LOG_EVERY_EVALS,
    workers=TRAIN_WORKERS,
    use_gpu=USE_GPU,
)

baseline_test_mse = None if result["baseline_test"] is None else result["baseline_test"]["relative_mse"]
learned_test_mse = None if result["test_metrics"] is None else result["test_metrics"]["relative_mse"]
external_holdout_mse = None if result.get("external_holdout_metrics") is None else result["external_holdout_metrics"]["relative_mse"]
generalization_delta = None if baseline_test_mse is None or learned_test_mse is None else (learned_test_mse - baseline_test_mse)
generalization_ok = None if generalization_delta is None else (generalization_delta <= 0.0)

{
    "preset_path": result["preset_path"],
    "report_path": result["report_path"],
    "runtime_sec": result.get("runtime_sec"),
    "train_relative_mse": result["train_metrics"]["relative_mse"],
    "baseline_test_relative_mse": baseline_test_mse,
    "learned_test_relative_mse": learned_test_mse,
    "generalization_delta": generalization_delta,
    "generalization_ok": generalization_ok,
}

## 4. Plot optimizer history


In [ ]:
history_rows = result.get("objective_history") or result.get("generation_history") or []
history_df = pd.DataFrame(history_rows)

if history_df.empty:
    print("No optimizer history rows captured.")
elif not {"iteration", "loss"}.issubset(history_df.columns):
    print(f"History columns missing: {list(history_df.columns)}")
else:
    history_df[["iteration", "loss"]].head()

In [ ]:
if history_df.empty or not {"iteration", "loss"}.issubset(history_df.columns):
    print("Skip plot: optimizer history is empty or missing required columns.")
else:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(history_df["iteration"], history_df["loss"], marker="o", linewidth=1.5)
    ax.set_title("Differential evolution objective history")
    ax.set_xlabel("iteration")
    ax.set_ylabel("relative MSE")
    plt.show()

## 5. Compare learned preset on the test slice


In [ ]:
test_rows = pd.DataFrame((result["test_metrics"] or {"rows": []})["rows"])
test_rows = test_rows.sort_values("rel_error", key=lambda s: s.abs(), ascending=False)
test_rows.head(10)


In [ ]:
if not test_rows.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(test_rows["actual_final"], test_rows["predicted_final"], alpha=0.8)
    diagonal = [
        test_rows[["actual_final", "predicted_final"]].min().min(),
        test_rows[["actual_final", "predicted_final"]].max().max(),
    ]
    axes[0].plot(diagonal, diagonal, linestyle="--", color="black")
    axes[0].set_title("Actual vs predicted")
    axes[0].set_xlabel("actual final")
    axes[0].set_ylabel("predicted final")

    axes[1].bar(test_rows["event_id"].astype(str), test_rows["rel_error"] * 100.0)
    axes[1].set_title("Relative error by event / snapshot")
    axes[1].set_xlabel("event id")
    axes[1].set_ylabel("error %")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No test rows available. Increase HISTORY_COUNT or lower similar_count.")
